# 2 - Finding the segments

K-Prototypes is the primary algorithm: the data is 6/9 categorical, and K-Means on one-hot columns makes the distance between any two professions a constant, so it cannot express that Doctor is nearer Healthcare than Artist.

With no label there is no answer key, so quality is established three ways that need none: internal validity (Gower silhouette), **resampling stability**, and agreement between structurally different algorithms.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, joblib
from IPython.display import Image, display
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from src import config as C, data as D, features as F, viz
viz.set_style()

from src import cluster as CL

In [ ]:
customers = D.load_customers()
scaled, raw, num_idx, cat_idx, encoder = F.build_cluster_frame(customers)
print(scaled.shape, "| categorical column indices:", cat_idx)

## Step 1 - internal validity across k

Cached by `run_all` because K-Prototypes over the whole k range takes several minutes.

In [ ]:
sel = joblib.load(C.PROCESSED / "cluster_selection.pkl")
sel

Every silhouette is low. That is the finding, not a failure: the customer base is a **gradient**, so no k produces well-separated clumps and the silhouette alone cannot choose one.

## Step 2 - bootstrap stability (the decisive check)

Resample the customers with replacement, recluster, and measure how often the same people stay together (Hennig 2007). A partition that dissolves under resampling is an artefact of this particular sample, whatever its silhouette says.

In [ ]:
per_cluster, summary = joblib.load(C.PROCESSED / "stability.pkl")
summary

In [ ]:
per_cluster

## Step 3 - the chosen k

In [ ]:
R = json.load(open(C.RESULTS_JSON, encoding="utf-8")) if False else None
import json
R = json.load(open(C.RESULTS_JSON, encoding="utf-8"))
print("k =", R["chosen_k"])
print(R["chosen_k_reason"])

In [ ]:
for name in ["04_cluster_selection", "05_cluster_stability"]:
    display(Image(filename=str(C.FIGURES / (name + ".png"))))

## The segments

In [ ]:
model, labels = joblib.load(C.PROCESSED / "kproto_labels.pkl")
prof = CL.profile_clusters(customers, labels)
prof.join(CL.distinguishing_features(customers, labels)["defining_traits"])

In [ ]:
for name in ["06_cluster_sizes", "07_cluster_personas",
             "08_cluster_projection"]:
    display(Image(filename=str(C.FIGURES / (name + ".png"))))

## Step 4 - do different algorithms agree?

Adjusted Rand Index here compares two **clusterings** with each other. No label is involved. If the structure were strong, structurally different algorithms would recover it.

In [ ]:
print(json.dumps(R["method_agreement"], indent=1))
display(Image(filename=str(C.FIGURES / "09_method_agreement.png")))